# Lesson 9: Loss (cross-entropy)

A loss is a single number that says how wrong a prediction is. Training is just changing the weights to make it smaller. For classification the standard loss is **cross-entropy**, which picks up right where softmax (Lesson 7) left off: it looks at the probability the model gave the correct class, and penalizes giving it too little.

Run every cell top to bottom, then do the exercises at the end and upload this notebook for feedback and a grade.

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

## Cross-entropy from scratch

Cross-entropy for one example is `-log(p)`, where `p` is the softmax probability the model put on the **true** class. Let us compute it by hand: softmax the logits, pick out the true class probability, take the negative log.

In [ ]:
logits = torch.tensor([2.0, 0.5, 0.1])   # raw scores for 3 classes
label = 0                                # the correct class is 0

probs = F.softmax(logits, dim=-1)
p_true = probs[label]
loss_by_hand = -torch.log(p_true)
print('probs:', probs)
print('p(true class):', p_true.item())
print('loss = -log(p_true):', loss_by_hand.item())

## In PyTorch: pass the logits, not the probabilities

`F.cross_entropy` (and the class `nn.CrossEntropyLoss`) takes the **raw logits** and the **integer** labels, and does the softmax and log for you, in a numerically safe way. So you do NOT softmax first. This is why Lesson 6 said to keep the logits.

In [ ]:
logits = torch.tensor([[2.0, 0.5, 0.1]])   # shape (batch=1, classes=3)
label = torch.tensor([0])                  # integer class, not one-hot

loss = F.cross_entropy(logits, label)
print('F.cross_entropy:', loss.item())
print('matches by-hand:', torch.allclose(loss, -torch.log(F.softmax(logits, dim=-1)[0, 0])))

## Confident and wrong is the worst case

The `-log` shape barely rewards being a little more right, but punishes confident mistakes without limit.

In [ ]:
good = F.cross_entropy(torch.tensor([[5.0, 0.0, 0.0]]), torch.tensor([0]))  # sure, and right
bad  = F.cross_entropy(torch.tensor([[0.0, 0.0, 5.0]]), torch.tensor([0]))  # sure, and wrong
print('confident + right:', good.item())   # tiny
print('confident + wrong:', bad.item())    # huge

## Over a batch

With several examples, cross-entropy computes each one's loss and returns the **average** by default: one number summarizing how wrong the model is across the batch. That is the number training will shrink.

In [ ]:
logits = torch.tensor([[2., 1., 0.],
                       [0., 3., 1.],
                       [1., 0., 4.]])
labels = torch.tensor([0, 1, 2])

print('mean loss:', F.cross_entropy(logits, labels).item())
print('per-example:', F.cross_entropy(logits, labels, reduction='none'))

## Why not just train on accuracy?

Accuracy is what you care about, but you cannot train on it: nudging a weight a hair usually flips no prediction, so accuracy gives no gradient to follow. Cross-entropy changes smoothly with every weight, so it always points toward a small improvement. You train on cross-entropy and report accuracy on the side (you will compute accuracy in Lesson 11).

## Exercises

Do these in the cells below, then upload the notebook. Only your code under each `# YOUR CODE HERE` line is graded.

In [ ]:
# E1: Use F.cross_entropy on logits = torch.tensor([[1.0, 3.0, 0.5]]) with
#     label = torch.tensor([1]). Print the loss.
# YOUR CODE HERE (write your answer below)


In [ ]:
# E2: Compute that same loss by hand (logits [[1.0, 3.0, 0.5]], true class 1): softmax the
#     logits, take the probability of the true class (index 1), and compute -log of it.
#     Confirm it matches E1 with torch.allclose.
# YOUR CODE HERE (write your answer below)


In [ ]:
# E3: Confident-correct vs confident-wrong. The true class is 1 for both. Compute
#     F.cross_entropy for logits_good = torch.tensor([[0.0, 4.0, 1.0]]) (class 1 wins) and
#     logits_bad  = torch.tensor([[4.0, 0.0, 1.0]]) (class 0 wins, but the truth is 1).
#     Print both losses and confirm the wrong one is much larger.
# YOUR CODE HERE (write your answer below)


In [ ]:
# E4: A batch. For logits = torch.tensor([[1., 2., 0.], [3., 0., 1.], [0., 1., 2.]]) and
#     labels = torch.tensor([1, 0, 2]), print the mean cross-entropy loss, and also the
#     per-example losses using reduction='none'.
# YOUR CODE HERE (write your answer below)
